# AI Image FFT Detector - Colab runner

Run setup, then only the authorized phase. Phase 1 collects original files; Phase 2 audits and freezes splits. No classifier training yet. CPU only; all code is on main. Save actual outputs for team review.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Clone the project
Clone main from the configured repository. An existing checkout is not overwritten; use a fresh session when you need a new checkout.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Rahul2-5/SpectraCheck.git"
PROJECT_DIR = Path("/content/ai-image-fft-detector")
if REPO_URL == "REPLACE_WITH_YOUR_GITHUB_REPOSITORY_URL":
    raise ValueError("Set REPO_URL to your GitHub repository URL first")
if PROJECT_DIR.exists():
    raise FileExistsError("Checkout already exists. Inspect it or use a fresh Colab session.")
subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
%pip install -r /content/ai-image-fft-detector/requirements.txt
%pip install --no-deps -e /content/ai-image-fft-detector


If Colab asks you to restart after installation, use Runtime > Restart session, then continue below (do not rerun the clone cell). This avoids mixing preloaded library versions with newly installed ones.

In [ ]:
from pathlib import Path
import importlib
import importlib.metadata as metadata
import os
import platform
import subprocess
import sys

PROJECT_DIR = Path("/content/ai-image-fft-detector")
os.chdir(PROJECT_DIR)
print("Python:", platform.python_version())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
modules = {"pillow": "PIL", "scikit-learn": "sklearn", "pyyaml": "yaml", "imagehash": "imagehash", "huggingface-hub": "huggingface_hub"}
for line in (PROJECT_DIR / "requirements.txt").read_text().splitlines():
    if not line.strip() or line.startswith("#"):
        continue
    package, expected = line.split("==")
    installed = metadata.version(package)
    assert installed == expected, f"Version mismatch: {package}: {installed} != {expected}"
    importlib.import_module(modules.get(package, package))
    print(f"{package}=={installed}")
from aidetect.io_utils import config_hash, load_config
cfg = load_config("config.yaml")
print("Config SHA-256:", config_hash(cfg))


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)


## Phase 0 review
Record setup results and check GitHub Actions. If Phase 1 has been authorized, continue below; otherwise stop here.

## Phase 1 - Original image collection
Download CIFAKE manually from [Kaggle](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images) and extract it on Drive. Set the folder below to the parent of train/REAL, train/FAKE, test/REAL, test/FAKE. The pilot is separate from main data. Keep this session on CPU.

In [ ]:
CIFAKE_SOURCE_DIR = Path("/content/drive/MyDrive/cifake")
if not CIFAKE_SOURCE_DIR.is_dir():
    raise FileNotFoundError("Set CIFAKE_SOURCE_DIR to your extracted CIFAKE folder on Drive")
subprocess.run([sys.executable, "scripts/download_pilot.py", "--source-dir", str(CIFAKE_SOURCE_DIR)], cwd=PROJECT_DIR, check=True)


### Main collection
The next command scans Tiny-GenImage metadata, counts every generator, then saves only SD15 images and a random sample of real images. It may transfer the full upstream Parquet dataset; a subset does not guarantee a small download. Rerun after a disconnection to reuse unchanged saved files. Exit code 2 means the SD15 count is too small: outputs are saved, but consult the supervisor before Phase 2.

In [ ]:
subprocess.run([sys.executable, "scripts/download_main.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
import json
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display

cfg = load_config(PROJECT_DIR / "config.yaml")
for collection in ("pilot", "main"):
    root = Path(cfg["paths"]["data_root"]) / collection
    manifest = pd.read_csv(root / "manifest.csv")
    record = json.loads((root / "acquisition.json").read_text())
    assert record["status"] == "complete", "Acquisition did not finish"
    listed = set(manifest["rel_path"])
    actual = {p.relative_to(root).as_posix() for folder in ("real", "ai") for p in (root / folder).glob("*") if p.is_file() and p.suffix != ".tmp"}
    assert len(manifest) == len(listed) and listed == actual, "Manifest/file counts differ"
    assert all((root / row.rel_path).stat().st_size == row.n_bytes for row in manifest.itertuples())
    print(collection, "saved counts:", manifest.groupby("label").size().to_dict())
    print(collection, "source generator counts:", record["generator_counts"])
    if collection == "main":
        assert record["saved_counts"]["ai"] >= cfg["dataset"]["minimum_ai_images"], "STOP: ask supervisor about insufficient SD15 data"
        print("Record SD15 and SD14 counts in the team tracker.")
    for label in (0, 1):
        samples = manifest[manifest["label"] == label].sample(n=5, random_state=cfg["seed"])
        for relative in samples["rel_path"]:
            with PILImage.open(root / relative) as image:
                image.load()
                print(collection, "label", label, relative, image.size, image.format)
                display(image.copy())


## Phase 1 review
Confirm generator counts, supervisor decision if SD15 is below the configured minimum, matching manifest/file counts, and five readable images per class. Continue only if Phase 2 is authorized. Keep real-data checks pending when their outputs are not available.

## Phase 2 - Audit before building the detector
Use the complete main collection already on Drive. Skip Phase 1 commands if it is already downloaded. The audit measures file format, quality and size, excludes undersized images before freezing, balances classes, and tests metadata-only shortcuts. It preserves originals. If a probe still exceeds 0.60 after the preplanned fix, STOP and review the report; do not weaken the threshold.

In [ ]:
subprocess.run([sys.executable, "scripts/run_audit.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
from IPython.display import Markdown, Image as DisplayImage, display

cfg = load_config(PROJECT_DIR / "config.yaml")
REPORTS_DIR = Path(cfg["paths"]["reports_root"])
report_text = (REPORTS_DIR / "audit_report.md").read_text()
display(Markdown(chr(10).join(line for line in report_text.splitlines() if not line.startswith("!["))))
for figure in sorted((REPORTS_DIR / "figures").glob("audit_*.png")):
    display(DisplayImage(filename=str(figure)))


In [ ]:
subprocess.run([sys.executable, "scripts/make_splits.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
import json
import pandas as pd
from aidetect.io_utils import sha256_of_file

DATA_ROOT = Path(cfg["paths"]["data_root"])
ARTIFACTS_DIR = Path(cfg["paths"]["artifacts_root"])
audit_result = json.loads((DATA_ROOT / "audit/audit_result.json").read_text())
assert audit_result["status"] == "passed"
assert audit_result["final_probe"]["max_accuracy"] <= cfg["audit"]["shortcut_max_accuracy"]
splits = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str, "group_id": str})
counts = splits.groupby(["split", "label"]).size().unstack(fill_value=0)
assert (counts[0] == counts[1]).all()
assert (splits.groupby("group_id")["split"].nunique() == 1).all()
for name, target in cfg["split"].items():
    fraction = (splits["split"] == name).mean()
    assert abs(fraction - target) <= cfg["splitting"]["proportion_tolerance"] + 1e-9
digest = sha256_of_file(DATA_ROOT / "splits.csv")
assert digest == (ARTIFACTS_DIR / "splits.sha256").read_text().strip()
print("Final metadata probes:", audit_result["final_probe"])
print("Class counts (real=0, AI=1):")
print(counts)
print("Groups crossing splits: 0")
print("Exclusions:", audit_result["exclusion_reasons"])
print("Splits SHA-256:", digest)


In [ ]:
import shutil

BACKUP_DIR = DATA_ROOT / "audit/review_backup"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
for filename in ("splits.sha256", "splits.json"):
    shutil.copy2(ARTIFACTS_DIR / filename, BACKUP_DIR / filename)
for source in REPORTS_DIR.glob("audit_*"):
    if source.is_file():
        shutil.copy2(source, BACKUP_DIR / source.name)
(BACKUP_DIR / "figures").mkdir(exist_ok=True)
for source in (REPORTS_DIR / "figures").glob("audit_*.png"):
    shutil.copy2(source, BACKUP_DIR / "figures" / source.name)
print("Review records backed up to Drive:", BACKUP_DIR)
print("After team review, commit actual artifacts/splits.sha256 and artifacts/splits.json to main.")


## STOP - Phase 2 team review
Review format/size/quality tables, both before/after probe scores, exclusions, exact class balance, and zero crossing duplicate groups. Record and commit the actual split hash/provenance. If blocked or infeasible, share the exact output; do not start Phase 3 or silently change the data. No frequency features have been computed.

## Phase 3 - Shared preprocessing
Run this after the reviewed Phase 2 split is present. The check uses only 20 deterministic training rows per class, displays successful preprocessed arrays, and counts every `smaller than target` rejection. It does not inspect validation/test rows or create feature files.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from aidetect.io_utils import load_config
from aidetect.preprocess import preprocess_image

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
active = pd.read_csv(DATA_ROOT / "audit/active_manifest.csv", dtype={"id": str})
frozen = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str})
train = active.merge(frozen[["id", "split"]], on="id", how="inner", validate="one_to_one")
train = train[train["split"] == "train"].sort_values(["label", "id"])
previews = {0: [], 1: []}
smaller_than_target = {0: 0, 1: 0}
for label in (0, 1):
    selected = train[train["label"] == label].head(20)
    if len(selected) != 20:
        raise ValueError(f"Need 20 training images for label {label}; found {len(selected)}")
    for row in selected.itertuples(index=False):
        try:
            array = preprocess_image(DATA_ROOT / row.rel_path, cfg)
        except ValueError as exc:
            if "smaller than target" not in str(exc):
                raise
            smaller_than_target[label] += 1
        else:
            previews[label].append((row.id, array))

for label, title in ((0, "real = 0"), (1, "AI-generated = 1")):
    figure, axes = plt.subplots(4, 5, figsize=(10, 10), squeeze=False)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, (identifier, array) in zip(axes.ravel(), previews[label]):
        axis.imshow(array, cmap="gray", vmin=0, vmax=1)
        axis.set_title(identifier, fontsize=7)
        axis.axis("off")
    figure.suptitle(title)
    plt.tight_layout()
    plt.show()

print("smaller than target counts:", smaller_than_target)

## Phase 4 - Frequency features
Run this after the reviewed Phase 2 split and Phase 3 preprocessing check. It uses 200 deterministic training rows per class only, plots mean log spectra and mean RAPSD with 95% bands, and saves figures under `reports/figures/`. It never reads validation/test rows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from aidetect.features import power_spectrum, rapsd
from aidetect.io_utils import load_config
from aidetect.preprocess import preprocess_image

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
REPORTS_DIR = PROJECT_DIR / cfg["paths"]["reports_root"]
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
active = pd.read_csv(DATA_ROOT / "audit/active_manifest.csv", dtype={"id": str})
frozen = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str})
train = active.merge(frozen[["id", "split"]], on="id", how="inner", validate="one_to_one")
train = train[train["split"] == "train"].sort_values(["label", "id"])
frequency = cfg.get("frequency", {})
epsilon = float(frequency.get("epsilon", 1.0e-12))
rings = int(cfg["rapsd_rings"])
spectra = {}
rapsd_values = {}
for label in (0, 1):
    selected = train[train["label"] == label].head(200)
    if len(selected) != 200:
        raise ValueError(f"Need 200 training images for label {label}; found {len(selected)}")
    class_spectra = []
    class_rapsd = []
    for row in selected.itertuples(index=False):
        image = preprocess_image(DATA_ROOT / row.rel_path, cfg)
        power = power_spectrum(image, window=frequency.get("hann_window", True))
        class_spectra.append(np.log10(np.maximum(power, epsilon)))
        class_rapsd.append(rapsd(power, rings=rings, epsilon=epsilon))
    spectra[label] = np.stack(class_spectra)
    rapsd_values[label] = np.stack(class_rapsd)

mean_spectra = {label: values.mean(axis=0) for label, values in spectra.items()}
vmin = min(values.min() for values in mean_spectra.values())
vmax = max(values.max() for values in mean_spectra.values())
figure, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for axis, label in zip(axes, (0, 1)):
    image = axis.imshow(mean_spectra[label], cmap="magma", vmin=vmin, vmax=vmax)
    axis.set_title("real = 0" if label == 0 else "AI-generated = 1")
    axis.set_xlabel("shifted FFT column")
    axis.set_ylabel("shifted FFT row")
figure.colorbar(image, ax=axes, label="mean log10 power")
figure.savefig(FIGURES_DIR / "phase_4_mean_log_spectrum.png", dpi=cfg["audit"]["figure_dpi"])
plt.show()

figure, axis = plt.subplots(figsize=(9, 5))
x_axis = np.arange(1, rings + 1)
for label, color, title in ((0, "tab:blue", "real = 0"), (1, "tab:orange", "AI-generated = 1")):
    values = rapsd_values[label]
    mean = values.mean(axis=0)
    band = 1.96 * values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
    axis.plot(x_axis, mean, color=color, label=title)
    axis.fill_between(x_axis, mean - band, mean + band, color=color, alpha=0.2)
axis.set_xlabel("RAPSD ring")
axis.set_ylabel("log10 mean power")
axis.set_title("Training-set mean RAPSD with 95% bands")
axis.legend()
figure.savefig(FIGURES_DIR / "phase_4_mean_rapsd.png", dpi=cfg["audit"]["figure_dpi"])
plt.show()
print("Saved:", FIGURES_DIR / "phase_4_mean_log_spectrum.png")
print("Saved:", FIGURES_DIR / "phase_4_mean_rapsd.png")

## Phase 5 - Pixel baseline and feature extraction
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 6 - Training and tuning
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 7 - Freeze and single test evaluation
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 8 - Local React prototype and Python prediction API
TODO: implement the React uploader and local Python API after authorization. The API must reuse shared aidetect preprocessing/features and the frozen primary model. Display calibrated results and spectra; keep uploads in memory. See docs/DECISIONS.md.

## Phase 9 - Report pack and reproducibility
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.